In [1]:
import pandas as pd
import kagglehub
import kaggle
import os

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Flatten
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam


Authentication required to call the Kaggle API.

First, you will need a Kaggle account. You can sign up at
  https://www.kaggle.com/account/login

Recommended: log in with OAuth via a web-based authorization flow.
No token to manage; credentials are cached locally for you.
    kaggle auth login

If you'd rather not use OAuth, generate an API token at
  https://www.kaggle.com/settings/api  (click "Generate New Token" under "API")
and supply it to the CLI in one of these ways:

  Option A: Environment variable
    export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx  # token copied from the settings UI

  Option B: API token file
    Save the token to ~/.kaggle/access_token


In [4]:
#import dataset
#KGAT_9f85a7deb631cf868d77dc993629a853
os.environ["KAGGLEHUB_CACHE_DIR"] = "D:\Datasets/kaggle_cache"
kagglehub.login()
path = kagglehub.competition_download('imagenet-object-localization-challenge')
print("Path to competition files:", path)

<>:3: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
C:\Users\josej\AppData\Local\Temp\ipykernel_3576\493295304.py:3: SyntaxWarning: invalid escape sequence '\D'
  os.environ["KAGGLEHUB_CACHE_DIR"] = "D:\Datasets/kaggle_cache"


Resuming download from 13705936896 bytes (152790735650 bytes left)...
Resuming download to C:\Users\josej\.cache\kagglehub\competitions\imagenet-object-localization-challenge.archive (13705936896/166496672546) bytes left.


  8%|▊         | 12.8G/155G [00:52<43:54:11, 966kB/s] 


KeyboardInterrupt: 

In [ ]:
def load_data(path, subset='training'):
    labels = pd.read_csv(path + 'labels.csv')
    
    data_datagen = ImageDataGenerator(
        rescale=1./255,
        validation_split=0.2)
    
    data_gen_flow = data_datagen.flow_from_dataframe(
        dataframe=labels,
        directory=path + 'final_files/',
        x_col='file_name',
        y_col='real_age',
        target_size=(224, 224),
        batch_size=16,
        class_mode='raw',
        subset=subset,
        seed=12345)

    return data_gen_flow

def create_model(input_shape):
    
    backbone = ResNet50(weights='imagenet', 
                        input_shape=input_shape,
                        include_top=False)

    model = Sequential()
    model.add(backbone)
    model.add(GlobalAveragePooling2D())
    model.add(Dense(1, activation='relu'))

    optimizer = Adam(learning_rate=0.0005)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

    return model

def train_model(model, train_data, test_data, batch_size=None, epochs=3):

    model.fit(train_data, validation_data=test_data, epochs=epochs, batch_size=batch_size, verbose=2) 
    

    return model
    
input_shape = (224, 224, 3)

train = load_data('/datasets/train/')
test = load_data('/datasets/test/', subset='testing')
model = create_model(input_shape)
model = train_model(model, train, test)

FileNotFoundError: [Errno 2] No such file or directory: '/datasets/train/labels.csv'